In [6]:
import sys
sys.path.append("..")

import pandas as pd
from src.daily_screener import screen_list

credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")

# Keep only gap-risk-clean stocks, take the top 15
clean_candidates = credit_spread_ranking[
    credit_spread_ranking["gap_risk_flag"].isna() | (credit_spread_ranking["gap_risk_flag"] == "")
]
top_15 = clean_candidates.head(15)

# Always include NIFTY and BANKNIFTY, gap-risk-checked the same way
must_include = credit_spread_ranking[credit_spread_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")]

final_candidates = pd.concat([top_15, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,credit_spread_score,gap_risk_flag,Total flips (closed),Median duration (days),% completed within a monthly cycle,Median max_adverse_pct,Worst max_adverse_pct (4yr),Worst single gap (4yr),current_trend_age_days,trend_age_note
0,NESTLEIND,91.5,NaN,54,24.2,59.3,2.72,5.96,4.37,33,NaN
1,ITC,86.9,NaN,52,23.7,61.5,2.41,9.56,4.68,7,NaN
2,SUNPHARMA,81.3,NaN,52,20.0,67.3,2.66,7.31,3.53,42,"note: aged trend — tested data shows momentum,..."
3,DRREDDY,80.8,NaN,57,22.2,68.4,2.84,7.88,4.27,25,NaN
4,ULTRACEMCO,80.4,NaN,66,19.0,66.7,2.39,7.21,3.61,49,"note: aged trend — tested data shows momentum,..."
5,COLPAL,79.8,NaN,58,19.2,67.2,3.10,5.84,2.32,33,NaN
6,HAVELLS,78.6,NaN,62,21.6,67.7,3.32,6.58,4.42,10,NaN
7,TECHM,78.5,NaN,64,19.6,62.5,3.16,6.85,4.37,31,NaN
8,ICICIPRULI,78.5,NaN,58,21.6,65.5,3.18,8.00,4.94,24,NaN
9,PIIND,78.1,NaN,50,22.1,64.0,3.50,8.76,4.07,14,NaN


In [7]:
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.fetch_data import fetch_historical_data, save_raw_data
from src.supertrend import apply_supertrend
from src.flip_analysis import detect_flips
from src.monthly_range import compute_monthly_range
from src.summary_stats import compute_summary_stats
from src.export_report import export_full_report
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        df = fetch_historical_data(kite, token, from_date=datetime(2022, 1, 1), to_date=datetime.now(), interval="4hour")
        save_raw_data(df, symbol)
        df = df.set_index("date")

        df_st = apply_supertrend(df)
        trades = detect_flips(df_st)
        monthly = compute_monthly_range(df)
        summary = compute_summary_stats(trades, monthly)
        export_full_report(trades, monthly, summary, symbol)

        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-04...
Saved 2261 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NESTLEIND_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NESTLEIND_full_report.xlsx
[1/16] NESTLEIND — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-04...
Saved 2261 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ITC_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/ITC_full_report.xlsx
[2/16] ITC — refreshed
Fetching 2022-01-01 to 2023-01-31...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-04...
Saved 2261 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/SUNPHARMA_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/SUNPHARMA_full_report.xlsx
[3/16] SUNPHARMA — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...


...

/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-04...
Saved 2261 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/BOSCHLTD_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/BOSCHLTD_full_report.xlsx
[15/16] BOSCHLTD — refreshed


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-04...
Saved 2261 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NIFTY_full_report.xlsx
[16/16] NIFTY — refreshed


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


In [8]:
from src.daily_screener import screen_list

status = screen_list(symbols_to_refresh)
status.sort_values("trend_age_days")

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct
2,SUNPHARMA,DOWN,0,20.0,False,False
5,COLPAL,DOWN,0,19.2,False,False
13,BANKNIFTY,UP,0,19.6,False,False
15,NIFTY,UP,6,15.2,False,False
1,ITC,UP,8,23.7,False,False
6,HAVELLS,UP,11,21.6,False,False
9,PIIND,UP,15,22.1,False,False
12,HDFCBANK,DOWN,15,17.2,False,False
8,ICICIPRULI,UP,25,21.6,True,False
3,DRREDDY,DOWN,26,22.2,True,False
